# Test de Tools de Reservas (Strands + DynamoDB)

Este notebook prueba las tools reales de `app/agent/tools.py` contra DynamoDB:
- `check_availability`
- `create_reservation`
- `list_reservations`
- `cancel_reservation`

Usa siempre un teléfono de pruebas para no mezclar con reservas reales.

## Requisitos

1. Tener `.env` configurado con AWS y DynamoDB.
2. Instalar dependencias del proyecto (`uv pip install -e .`).
3. Ejecutar el notebook desde la raíz del repo.

In [1]:
import json
import os
import sys
from datetime import datetime, timedelta
from pathlib import Path

from dotenv import load_dotenv

def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'pyproject.toml').exists() and (candidate / 'app').is_dir():
            return candidate
    raise RuntimeError(
        f'No se encontró la raíz del repo desde {start}. '
        'Abre el notebook dentro del proyecto o ajusta manualmente ROOT.'
    )

ROOT = find_repo_root(Path.cwd())
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

load_dotenv(ROOT / '.env')

valid_bool_values = {'true', 'false', '1', '0', 'yes', 'no', 'on', 'off'}
debug_env = (os.getenv('DEBUG') or '').strip().lower()
if debug_env and debug_env not in valid_bool_values:
    print("WARNING: DEBUG inválido en .env; usando DEBUG=false para esta sesión de notebook")
    os.environ['DEBUG'] = 'false'

print('Repo root:', ROOT)
print('Dynamo table:', os.getenv('DYNAMODB_TABLE_NAME'))
print('AWS region:', os.getenv('DYNAMODB_REGION') or os.getenv('AWS_REGION'))


Repo root: /mnt/c/Users/eduardo.merino/documents/freelance/reservai-demo-strands-agents
Dynamo table: reservai-demo-reservations
AWS region: eu-west-1


In [2]:
from app.agent.tools import (
    check_availability,
    create_reservation,
    list_reservations,
    cancel_reservation,
)
from app.reservations.service import WEEKLY_SERVICE_WINDOWS

def parse_tool_result(result: dict) -> dict:
    text = result['content'][0]['text']
    return json.loads(text)

def pretty(data: dict):
    print(json.dumps(data, indent=2, ensure_ascii=False))

In [3]:
# Config de prueba (puedes cambiar estos valores)
TEST_PHONE = os.getenv('TEST_RESERVATION_PHONE', '34600000000')
TEST_NAME = os.getenv('TEST_RESERVATION_NAME', 'Test Notebook')
TEST_PARTY_SIZE = int(os.getenv('TEST_RESERVATION_PARTY_SIZE', '2'))
TEST_NOTES = 'Reserva creada desde notebook de pruebas'

def next_open_date(start_date: datetime | None = None) -> str:
    base = start_date or datetime.now()
    for offset in range(14):
        candidate = (base + timedelta(days=offset)).date()
        if WEEKLY_SERVICE_WINDOWS.get(candidate.weekday()):
            return candidate.isoformat()
    raise RuntimeError('No se encontró ningún día abierto en los próximos 14 días')

TARGET_DATE = next_open_date()
print('TEST_PHONE =', TEST_PHONE)
print('TARGET_DATE =', TARGET_DATE)

TEST_PHONE = 34600000000
TARGET_DATE = 2026-03-03


In [4]:
# 1) Disponibilidad
availability_raw = check_availability(
    date=TARGET_DATE,
    party_size=TEST_PARTY_SIZE,
)
availability = parse_tool_result(availability_raw)
pretty(availability)

if not availability.get('available') or not availability.get('times'):
    raise RuntimeError('No hay disponibilidad para probar create_reservation')

TARGET_TIME = availability['times'][0]['time']
print('TARGET_TIME =', TARGET_TIME)

{
  "date": "2026-03-03",
  "party_size": 2,
  "preferred_time": null,
  "duration_minutes": 90,
  "available": true,
  "total_open_slots": 54,
  "total_bookable_options": 54,
  "times": [
    {
      "time": "13:00",
      "end_time": "14:30",
      "free_tables": 6,
      "example_tables": [
        "Mesa 1",
        "Mesa 2",
        "Mesa 3"
      ]
    },
    {
      "time": "13:30",
      "end_time": "15:00",
      "free_tables": 6,
      "example_tables": [
        "Mesa 1",
        "Mesa 2",
        "Mesa 3"
      ]
    },
    {
      "time": "14:00",
      "end_time": "15:30",
      "free_tables": 6,
      "example_tables": [
        "Mesa 1",
        "Mesa 2",
        "Mesa 3"
      ]
    },
    {
      "time": "14:30",
      "end_time": "16:00",
      "free_tables": 6,
      "example_tables": [
        "Mesa 1",
        "Mesa 2",
        "Mesa 3"
      ]
    },
    {
      "time": "20:00",
      "end_time": "21:30",
      "free_tables": 6,
      "example_tables": [
        "

In [5]:
# 2) Crear reserva
create_raw = create_reservation(
    phone=TEST_PHONE,
    customer_name=TEST_NAME,
    date=TARGET_DATE,
    time=TARGET_TIME,
    party_size=TEST_PARTY_SIZE,
    notes=TEST_NOTES,
)
create_data = parse_tool_result(create_raw)
pretty(create_data)

RESERVATION_ID = create_data.get('reservation_id')
if not RESERVATION_ID:
    raise RuntimeError('No se devolvió reservation_id al crear reserva')

print('RESERVATION_ID =', RESERVATION_ID)

{
  "reservation_id": "rsv_20260302_7293b7e9",
  "status": "ACTIVE",
  "name": "Test Notebook",
  "phone": "34600000000",
  "date": "2026-03-03",
  "time": "13:00",
  "end_time": "14:30",
  "duration_minutes": 90,
  "party_size": 2,
  "table": "Mesa 1",
  "area": "interior",
  "notes": "Reserva creada desde notebook de pruebas",
  "ok": true
}
RESERVATION_ID = rsv_20260302_7293b7e9


In [6]:
# 3) Listar reservas activas del teléfono
list_raw = list_reservations(
    phone=TEST_PHONE,
    only_active=True,
    limit=10,
)
list_data = parse_tool_result(list_raw)
pretty(list_data)

{
  "phone": "34600000000",
  "only_active": true,
  "count": 1,
  "reservations": [
    {
      "reservation_id": "rsv_20260302_7293b7e9",
      "status": "ACTIVE",
      "name": "Test Notebook",
      "date": "2026-03-03",
      "time": "13:00",
      "end_time": "14:30",
      "duration_minutes": 90,
      "party_size": 2,
      "table": "Mesa 1"
    }
  ],
  "ok": true
}


In [7]:
# 4) Cancelar reserva
cancel_raw = cancel_reservation(
    phone=TEST_PHONE,
    reservation_id=RESERVATION_ID,
)
cancel_data = parse_tool_result(cancel_raw)
pretty(cancel_data)

{
  "reservation_id": "rsv_20260302_7293b7e9",
  "status": "CANCELLED",
  "name": "Test Notebook",
  "date": "2026-03-03",
  "time": "13:00",
  "end_time": "14:30",
  "duration_minutes": 90,
  "party_size": 2,
  "table": "Mesa 1",
  "cancelled_at": "2026-03-02T17:13:11+01:00",
  "ok": true
}


In [8]:
# 5) Verificar que no quede activa (opcional)
post_list_raw = list_reservations(
    phone=TEST_PHONE,
    only_active=True,
    limit=10,
)
post_list_data = parse_tool_result(post_list_raw)
pretty(post_list_data)

{
  "phone": "34600000000",
  "only_active": true,
  "count": 0,
  "reservations": [],
  "ok": true
}
